# Chương 6. Tư duy bảng dữ liệu với pandas

**Câu hỏi mở đầu:** Khi nhận một tập dữ liệu hoàn toàn mới, điều gì cần được kiểm tra trước?

Notebook này là tài nguyên đồng hành của chương. Mỗi phần đều đi theo nhịp **câu hỏi → dữ liệu → mã → kết quả → diễn giải → kiểm tra bằng chứng**.

## Mục tiêu

- Tái hiện các ví dụ cốt lõi của chương bằng mã có thể chạy lại.
- Kiểm tra giả định trước khi diễn giải output.
- Kết thúc bằng ít nhất một câu hỏi về điều mà kết quả **chưa** cho biết.

In [ ]:
from pathlib import Path
import warnings
warnings.filterwarnings("error", category=FutureWarning)
warnings.filterwarnings("error", category=DeprecationWarning)
ROOT = Path.cwd()
DATA = ROOT / "data"
print("Working root:", ROOT)


In [ ]:
import pandas as pd
import numpy as np
pd.set_option("display.max_columns", 20)


## 1. Rà soát dữ liệu mới: Cấu trúc → Kiểu → Đầy đủ → Hợp lý

In [ ]:
df = pd.read_csv(DATA / "metromart" / "metromart_sales_raw.csv")
print("shape:", df.shape)
display(df.head())
print(df.dtypes)

## 2. Quản lý bộ nhớ khi dữ liệu lớn

pandas chủ yếu làm việc **trong RAM**. Trước khi tối ưu, hãy đo bộ nhớ thật sự đang dùng.

Ba ý cần kiểm tra:

1. Cột phân loại lặp ít nhãn có phù hợp với `category` không?
2. Kiểu số có đang rộng hơn mức cần thiết không?
3. Nếu dữ liệu vẫn vượt RAM, có thể đọc ít cột hơn hoặc xử lý theo `chunksize` không?

> Lưu ý: pandas 3.x có dtype chuỗi `str` mặc định. Không nên hiểu quy tắc này thành “mọi cột chữ phải đổi từ object sang category”.

In [ ]:
before_mb = df.memory_usage(deep=True).sum() / 1024**2
print(f"Memory before: {before_mb:.2f} MB")

memory_by_col = (
    df.memory_usage(deep=True)
      .sort_values(ascending=False)
      .div(1024**2)
)
display(memory_by_col.rename("MB").to_frame())

### Tối ưu có kiểm tra

Với MetroMart, các biến như `store_type`, `region`, `payment_method` và `transaction_type` có ít mức lặp lại nên là ứng viên tự nhiên cho `category`. Ngược lại, `transaction_id` gần như duy nhất cho từng hàng nên không nên chuyển một cách máy móc.

`float32` dùng khoảng một nửa bộ nhớ của `float64`, nhưng có độ chính xác thấp hơn. Vì vậy, ta chỉ downcast những cột mà miền giá trị và yêu cầu độ chính xác cho phép.

In [ ]:
optimized = df.copy()

cat_cols = [
    "store_id", "store_type", "region",
    "product_id", "category",
    "payment_method", "transaction_type",
]
for col in cat_cols:
    optimized[col] = optimized[col].astype("category")

optimized["quantity"] = pd.to_numeric(
    optimized["quantity"], downcast="integer"
)
optimized["discount_pct"] = pd.to_numeric(
    optimized["discount_pct"], downcast="float"
)

after_mb = optimized.memory_usage(deep=True).sum() / 1024**2

print(f"Memory before : {before_mb:.2f} MB")
print(f"Memory after  : {after_mb:.2f} MB")
print(f"Reduction     : {(1 - after_mb / before_mb) * 100:.1f}%")
print()
print(optimized.dtypes)

assert np.allclose(
    df["discount_pct"].to_numpy(),
    optimized["discount_pct"].to_numpy(),
    equal_nan=True,
)
assert (df["quantity"] == optimized["quantity"]).all()

### Nếu dữ liệu vẫn vượt RAM

Tối ưu dtype không giải quyết mọi bài toán. Khi cần, hãy:

- dùng `usecols=` để chỉ đọc các cột cần thiết;
- khai báo `dtype=` ngay khi đọc;
- dùng `chunksize=` cho các phép toán có thể xử lý độc lập theo khối;
- chuyển sang cơ sở dữ liệu hoặc công cụ hỗ trợ xử lý ngoài bộ nhớ/phân tán nếu phép toán toàn cục vẫn vượt RAM.

**Kiểm tra bằng chứng:** tiết kiệm RAM chỉ là cải thiện tài nguyên tính toán; nó không làm dữ liệu “đúng hơn”.

## 3. Tình trạng thiếu dữ liệu

In [ ]:
missing = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_rate": df.isna().mean(),
}).sort_values("missing_rate", ascending=False)
display(missing.head(8))

## 4. Nhãn và phạm vi đáng kiểm tra

In [ ]:
print("category variants:")
print(df["category"].value_counts().head(12))
print("quantity range:", df["quantity"].min(), df["quantity"].max())
print("discount max:", df["discount_pct"].max())

### Bản ghi nhớ kiểm tra dữ liệu

- **Cấu trúc:** 18.000 hàng, 13 cột.
- **Kiểu:** một số cột ID được lưu dạng chuỗi; kiểu lưu trữ không quyết định ý nghĩa phân tích.
- **Đầy đủ:** `customer_id` và `unit_price` có giá trị thiếu.
- **Hợp lý:** quantity âm và discount lớn hơn 1 cần được chẩn đoán theo ngữ cảnh.

### Kiểm tra bằng chứng

Từ output trên chỉ có thể nói “có giá trị cần kiểm tra”; chưa thể kết luận nguyên nhân hoặc quyết định cách sửa.

## Thực hành

Viết Data Audit Memo gồm ba cột: **Phát hiện — Bằng chứng — Cần kiểm tra tiếp**. Chưa được gọi `dropna()` hoặc `fillna()`.

---
### Bạn đã sẵn sàng sang chương tiếp theo nếu có thể…

- giải thích output bằng lời;
- chỉ ra ít nhất một giả định;
- nói được kết quả chưa cho phép kết luận điều gì.

**Exit check:** Nếu mã chạy không lỗi nhưng câu trả lời trái với ý nghĩa của dữ liệu, bạn sẽ kiểm tra điều gì trước?